In [14]:
!pip install requests==2.32.5

!pip install -U langchain langchain-community faiss-cpu
!apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 8.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
google-adk 1.27.1 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.40.0 which is incompatible.
google-adk 1.27.1 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requi

In [15]:
import subprocess, time
subprocess.Popen(['ollama', 'serve'])
time.sleep(5)
print("Ollama Start")

Ollama Start


In [16]:
!pip install ollama

In [17]:
!ollama pull llama3.2
!ollama pull mxbai-embed-large

In [18]:
!ollama list

NAME                        ID              SIZE      MODIFIED               
mxbai-embed-large:latest    468836162de7    669 MB    Less than a second ago    
llama3.2:latest             a80c4f17acd5    2.0 GB    1 second ago              
mistral:latest              6577803aa9a0    4.4 GB    17 minutes ago            


In [19]:
documents = ["Shelf Discovery Tourism: Exploring local artisanal markets and specialty grocery stores for unique souvenirs. cost_rating:1, interest_socre: 92, Category: Cultural" ,
 "Urban Micro-Hiking: Using local green belts and hillocks for sunrise 'run-cations' followed by local breakfast. cost_rating:3, interest_socre: 85, Category: Fitness",
"Backyard Reading Retreat: Setting up a dedicated 'phone-free' zone with a library-loaned book list and DIY cold-brew station.. cost_rating:1, interest_socre: 78, Category: Relaxation",
"Swimming competion: Learn a new skill how to swim first time and first time win a swimming compation. cost_rating = 2, intrest_score:90, Category: Fitness"]

In [20]:
# generate embeddings:
from langchain_community.embeddings import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="mxbai-embed-large")

/tmp/ipykernel_4687/742683725.py:4: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="mxbai-embed-large")


In [32]:
# Vectore DB Create:
from langchain_community.vectorstores import FAISS

db = FAISS.from_texts(documents, embeddings)
db.save_local("my_vector_db")

In [22]:
# Load LLM:
from langchain_community.llms import Ollama
llm = Ollama(model="llama3.2")

/tmp/ipykernel_4687/1304135915.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llama3.2")


In [23]:
# User Query:
query = input("Enter your question: ")

Enter your question: which place is best gujrat or surat for summer vacation


In [24]:
# similarity search:
context = db.similarity_search(query, k=3)
context

[Document(id='9f3641b1-dcbd-4730-902d-a39729ff7c1a', metadata={}, page_content='Shelf Discovery Tourism: Exploring local artisanal markets and specialty grocery stores for unique souvenirs. cost_rating:1, interest_socre: 92, Category: Cultural'),
 Document(id='1dbcbc68-d241-4281-8b73-5d7ee8ea4bcd', metadata={}, page_content="Urban Micro-Hiking: Using local green belts and hillocks for sunrise 'run-cations' followed by local breakfast. cost_rating:3, interest_socre: 85, Category: Fitness"),
 Document(id='dcfed1fd-9c89-451d-8d84-a3be30a2260e', metadata={}, page_content="Backyard Reading Retreat: Setting up a dedicated 'phone-free' zone with a library-loaned book list and DIY cold-brew station.. cost_rating:1, interest_socre: 78, Category: Relaxation")]

In [25]:
# Prompt:
prompt = """
  You are an expert vacation guide for local tourism.
You are professional in your work and highly skilled in recommending summer activities.
Your focus is to suggest activities that are low-cost and highly interesting.
Use the context provided below to generate your answer.

  context: {context}
  question: {query}
  answer:
"""

In [26]:
response = llm.invoke(prompt.format(context=context, query=query))
print(response)

As your go-to Vacation Guide, I'd be delighted to help you plan an unforgettable summer getaway in Gujarat or Surat!

Considering your interests in low-expenses and high-interest activities, I've shortlisted the top recommendations for each destination:

**Gujarat:**

1. **Dholavira**: Explore this ancient Indus Valley Civilization site, a UNESCO World Heritage Site, by visiting the nearby town of Bhuj. Enjoy a sunrise 'run-cation' (Urban Micro-Hiking) followed by a local breakfast at a nearby café.
2. **Palitana**: Visit the Jain Temples of Palitana, a series of intricately carved structures that offer breathtaking views of the surrounding landscape. Take a self-guided tour and enjoy the scenic surroundings without spending a fortune.
3. **Kirti Stambh**: Head to Kirti Stambh, a 12th-century Jain monument, for a relaxing Backyard Reading Retreat experience. Set up your phone-free zone with a book from the local library and enjoy a refreshing cold-brew station.

**Surat:**

1. **Dugesh